In [ ]:
# @title 1. Setup and Library Imports
# This cell imports all the necessary libraries for our operations.
import ipywidgets as widgets
from IPython.display import display, FileLink
import zipfile
from PIL import Image
import io
import os

print("Libraries imported successfully!")

In [ ]:
# @title 2. Upload Your Image Zip File
# This cell creates a file upload widget.
# Click the "Upload" button and select the zip file from your computer.

uploader = widgets.FileUpload(
    accept='.zip',
    description='Upload Zip',
    button_style='info'
)

display(uploader)

In [ ]:
# @title 3. Process Images and Create New Zip Files
# This is the main part of our notebook.
# It will:
# - Read the uploaded zip file.
# - Crop the images in two different ways.
# - Create two new zip files with the results.

def process_images(upload):
    if not upload:
        print("Please upload a zip file first.")
        return

    # Get the uploaded file's content
    uploaded_filename = next(iter(upload))
    zip_content = upload[uploaded_filename]['content']

    # Define the names for our output zip files
    right_half_zip_name = 'cropped_right_half.zip'
    top_right_quadrant_zip_name = 'cropped_top_right_quadrant.zip'

    # Create in-memory zip files to store the cropped images
    right_half_zip_buffer = io.BytesIO()
    top_right_quadrant_zip_buffer = io.BytesIO()

    with zipfile.ZipFile(io.BytesIO(zip_content), 'r') as original_zip:
        with zipfile.ZipFile(right_half_zip_buffer, 'w') as right_half_zip, \
             zipfile.ZipFile(top_right_quadrant_zip_buffer, 'w') as top_right_quadrant_zip:

            for item in original_zip.infolist():
                if not item.is_dir() and item.filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')):
                    try:
                        with original_zip.open(item) as image_file:
                            img = Image.open(image_file)
                            width, height = img.size

                            # --- Crop Right Half ---
                            right_half_coords = (width / 2, 0, width, height)
                            cropped_right = img.crop(right_half_coords)
                            # Save to a byte buffer
                            right_buffer = io.BytesIO()
                            cropped_right.save(right_buffer, format=img.format or 'PNG')
                            right_half_zip.writestr(f"right_half_{item.filename}", right_buffer.getvalue())

                            # --- Crop Top Right Quadrant ---
                            top_right_coords = (width / 2, 0, width, height / 2)
                            cropped_top_right = img.crop(top_right_coords)
                            # Save to a byte buffer
                            top_right_buffer = io.BytesIO()
                            cropped_top_right.save(top_right_buffer, format=img.format or 'PNG')
                            top_right_quadrant_zip.writestr(f"top_right_{item.filename}", top_right_buffer.getvalue())

                    except Exception as e:
                        print(f"Could not process file {item.filename}: {e}")

    # Save the in-memory zip files to disk
    with open(right_half_zip_name, 'wb') as f:
        f.write(right_half_zip_buffer.getvalue())
    with open(top_right_quadrant_zip_name, 'wb') as f:
        f.write(top_right_quadrant_zip_buffer.getvalue())

    print("Processing complete!")
    print(f"Created '{right_half_zip_name}' and '{top_right_quadrant_zip_name}'.")

    # --- Display Download Links ---
    print("\nDownload your new zip files:")
    display(FileLink(right_half_zip_name))
    display(FileLink(top_right_quadrant_zip_name))


# Run the processing function with the uploaded file
process_images(uploader.value)